In [ ]:
import subprocess

def check_r_version():
    try:
        output = subprocess.check_output(['R', '--version'], universal_newlines=True)
        version_line = next((line for line in output.splitlines() if 'version' in line), None)
        if version_line:
            version = version_line.strip()
            return version
        else:
            return None
    except FileNotFoundError:
        return None

r_version = check_r_version()
if r_version:
    print(f"R version: {r_version}")
    # Proceed with rpy2 installation if version is compatible
else:
    print("R not found or version could not be determined.")


In [ ]:
import os
import shutil
import platform

print("=== R and rpy2 Setup - Final Solution ===")

# R installation paths
r_home = "C:/Program Files (x86)/R/R-4.6.1"
r_bin_main = os.path.join(r_home, "bin")
r_bin_64 = os.path.join(r_home, "bin", "x64")

# Source and target for R.dll
r_dll_source = os.path.join(r_bin_64, "R.dll")
r_dll_target = os.path.join(r_bin_main, "R.dll")

print(f"R installation: {r_home}")
print(f"Python: {platform.architecture()[0]}")

# Set up environment
os.environ['R_HOME'] = r_home
os.environ['R_ARCH'] = '/x64'
current_path = os.environ.get('PATH', '')
os.environ['PATH'] = r_bin_64 + os.pathsep + r_bin_main + os.pathsep + current_path

try:
    os.add_dll_directory(r_bin_64)
    os.add_dll_directory(r_bin_main)
except Exception:
    pass

# The definitive fix: Copy R.dll and dependencies
if os.path.exists(r_dll_source):
    print(f"✓ Found R.dll at: {r_dll_source}")
    
    # Check if we need to copy
    if not os.path.exists(r_dll_target):
        print("Copying R.dll to expected location...")
        
        # Files to copy from x64 to main bin
        files_to_copy = [
            "R.dll",
            "Rblas.dll", 
            "Rlapack.dll",
            "Riconv.dll"
        ]
        
        try:
            for filename in files_to_copy:
                src = os.path.join(r_bin_64, filename)
                dst = os.path.join(r_bin_main, filename)
                
                if os.path.exists(src) and not os.path.exists(dst):
                    shutil.copy2(src, dst)
                    print(f"  ✓ Copied {filename}")
                elif os.path.exists(dst):
                    print(f"  ✓ {filename} already exists")
                    
        except PermissionError:
            print("❌ Permission denied. Run this as administrator:")
            print(f'PowerShell command (run as admin):')
            for filename in files_to_copy:
                src = os.path.join(r_bin_64, filename).replace('/', '\\')
                dst = os.path.join(r_bin_main, filename).replace('/', '\\')
                if os.path.exists(src.replace('\\', '/')):
                    print(f'Copy-Item "{src}" "{dst}"')
            
            print("\nAlternatively, restart VS Code as administrator and run this cell again.")
            
        except Exception as e:
            print(f"❌ Copy failed: {e}")
    else:
        print("✓ R.dll already in correct location")

# Test if the fix worked
print("\n=== Testing rpy2 ===")
try:
    import rpy2.robjects as robjects
    print("✓ rpy2.robjects imported successfully!")
    
    # Test basic R operations
    result = robjects.r('1 + 1')
    print(f"✓ Test calculation: 1 + 1 = {result[0]}")
    
    r_version = robjects.r('R.version.string')
    print(f"✓ R version: {r_version[0]}")
    
    print("\n🎉 SUCCESS! R integration with Python is working!")
    print("\nYou can now use R in Python like this:")
    print("import rpy2.robjects as robjects")
    print("result = robjects.r('your_r_code_here')")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Install rpy2: pip install rpy2")
    
except Exception as e:
    print(f"❌ Runtime error: {e}")
    
    if "R.dll" in str(e):
        print("\n⚠ R.dll still not found. Manual fix required:")
        print("Run PowerShell as Administrator and execute:")
        print(f'Copy-Item "{r_dll_source}" "{r_dll_target}"')
        
        print("\nOr try reinstalling rpy2 with R_HOME set:")
        print(f"$env:R_HOME = '{r_home}'")
        print("pip uninstall rpy2")
        print("pip install rpy2")

print(f"\n=== Final Status ===")
print(f"R.dll exists at expected location: {os.path.exists(r_dll_target)}")
print(f"R_HOME: {os.environ.get('R_HOME')}")
print(f"Architecture match: Python {platform.architecture()[0]}, R installation in x64 folder")


In [ ]:
#Run as administrator
import rpy2.robjects as robjects
# Define an R function
r_code = """
sum_r <- function(x, y) {
return(x + y)
}
"""
# Execute the R code
robjects.r(r_code)
# Access the R function from Python
sum_r = robjects.globalenv['sum_r']
# Call the R function with arguments
result = sum_r(3, 5)
print(result[0]) 

In [ ]:
#Run as administrator
import rpy2.robjects as robjects
# Define an R function
r_code = """
print(2+2)
print(2*2)
library(haven)
"""
# Execute the R code
robjects.r(r_code)

In [ ]:
import pandas as pd
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

# Define the R interface object once
r = ro.r

# Python DataFrame
df = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie"],
    "age": [25, 30, 35],
    "salary": [50000, 60000, 70000]
})

# Define R function
r("""
addtwo <- function(df) {
    df$salary2 <- df$salary + 2
    return(df)
}
""")

# Convert Python DataFrame to R DataFrame with a local conversion context
with localconverter(ro.default_converter + pandas2ri.converter):
    rdf = pandas2ri.py2rpy(df)

# Call the R function
r.assign("rdf", rdf)
r("rdf <- addtwo(rdf)")

# Convert back to Python DataFrame
with localconverter(ro.default_converter + pandas2ri.converter):
    df_result = ro.conversion.rpy2py(r("rdf"))

# Safety check in case it's already converted
if not isinstance(df_result, pd.DataFrame):
    df_result = pd.DataFrame(df_result)

print(df_result)


In [ ]:
#library(haven) # import library
#print(2 + 2) # execute command
#print(paste("Finished at",
#            format(Sys.time(), "%Y-%m-%d %H:%M:%S"))) #dateandtimeofexecution

import subprocess
result = subprocess.run(
    [r"C:\Program Files (x86)\R\R-4.6.1\bin\Rscript.exe",
     r"C:\Language_Interfacing\code\input_code\addtwo.R"],
    capture_output=True, text=True
)
print(result.stdout)




In [ ]:
from git import Repo
from pathlib import Path

repo_root = Path(r"C:\Language_Interfacing")
repo = Repo(repo_root)

# Ensure the Stata data file is ignored in git
gitignore_path = repo_root / ".gitignore"
ignore_entries = ["_load.matrix.C.dta", ".RData", ".Rhistory"]


current_text = gitignore_path.read_text(encoding="utf-8") if gitignore_path.exists() else ""
existing_lines = {line.strip() for line in current_text.splitlines()}

for ignore_entry in ignore_entries:
    if ignore_entry not in existing_lines:
        separator = "" if (not current_text or current_text.endswith("\n")) else "\n"
        gitignore_path.write_text(f"{current_text}{separator}{ignore_entry}\n", encoding="utf-8")
        print(f"Added to .gitignore: {ignore_entry}")
    else:
        print(f"Already in .gitignore: {ignore_entry}")

repo.index.add([r"code\R_within_Python.ipynb"])
repo.index.add([r"code\input_code\addtwo.R"])
print("Staged: code\\R_within_Python.ipynb")
print("Staged: code\\input_code\\addtwo.R")

if repo.is_dirty(index=True, working_tree=True, untracked_files=False):
    commit_message = "R within Python notebook"
    commit = repo.index.commit(commit_message)
    print(f"Committed: {commit.hexsha[:7]} - {commit_message}")
else:
    print("No tracked changes to commit")

origin = repo.remote(name="origin")
push_info = origin.push("main")[0]
print(f"Pushed to origin/main: {push_info.summary}")